# Phase 10.1-10.2: Runtime Scaling Analysis

Measure computational scaling behavior of Markovianity diagnostic methods across data sizes and parallelization configurations.

## Workflow
1. Generate synthetic order-1 Markov data with varying T (time series length) and d (dimensionality)
2. Run cgc and fcgc analysis at each data size with p_values in [5, 10]
3. Measure wall-clock runtime with n_jobs in [1, 4]
4. Export raw measurements to runtime_grid.csv
5. Create scaling visualization (runtime_plot.png)
6. Export analysis metadata (manifest.json)

## Output Directory
outputs/power/runtime_scaling/

### Setup: Imports and Project Root

In [ ]:
from __future__ import annotations

import sys
import json
import logging
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

PROJECT_ROOT = Path('.').resolve()
while not (PROJECT_ROOT / 'src' / 'markovianity_diagnostic').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise FileNotFoundError("Could not find project root")

sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from markovianity_diagnostic.experiments.simulations import scenario_order1_unconfounded
from markovianity_diagnostic.experiments.adapters import (
    analyze_with_gcstar_cgc,
    analyze_with_gcstar_fcgc,
)

logger.info(f"Project root: {PROJECT_ROOT}")
print(f"Project root: {PROJECT_ROOT}")

### Configuration

In [ ]:
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'power' / 'runtime_scaling'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Measurement parameters
T_VALUES = [500, 1000, 2000, 5000]
D_VALUES = [5, 10, 20, 50]
METHODS = ['cgc', 'fcgc']
P_GRID_MAX_VALUES = [5, 10]
N_JOBS_VALUES = [1, 4]

# Random seed for reproducibility
SEED = 42

logger.info(f"Output directory: {OUTPUT_DIR}")
logger.info("Measurement grid configuration:")
logger.info(f"  T (time series length): {T_VALUES}")
logger.info(f"  d (dimensionality): {D_VALUES}")
logger.info(f"  Methods: {METHODS}")
logger.info(f"  p_grid_max (max p value): {P_GRID_MAX_VALUES}")
logger.info(f"  n_jobs (parallelization): {N_JOBS_VALUES}")
logger.info(f"  Total combinations: {len(T_VALUES) * len(D_VALUES) * len(METHODS) * len(P_GRID_MAX_VALUES) * len(N_JOBS_VALUES)}")

print(f"Output directory: {OUTPUT_DIR}")
print(f"\nMeasurement grid:")
print(f"  T (time series length): {T_VALUES}")
print(f"  d (dimensionality): {D_VALUES}")
print(f"  Methods: {METHODS}")
print(f"  p_grid_max (max p value): {P_GRID_MAX_VALUES}")
print(f"  n_jobs (parallelization): {N_JOBS_VALUES}")
print(f"\nTotal combinations: {len(T_VALUES) * len(D_VALUES) * len(METHODS) * len(P_GRID_MAX_VALUES) * len(N_JOBS_VALUES)}")

In [ ]:
# Check for cached outputs
expected_outputs = {
    'runtime_grid.csv': OUTPUT_DIR / 'runtime_grid.csv',
    'runtime_plot.png': OUTPUT_DIR / 'runtime_plot.png',
    'manifest.json': OUTPUT_DIR / 'manifest.json',
}

outputs_exist = all(fpath.exists() for fpath in expected_outputs.values())

if outputs_exist:
    logger.info("✅ Outputs already exist - loading cached results")
    print("✅ Outputs already exist - loading cached results")
    print(f"Output directory: {OUTPUT_DIR}")
    for name, path in expected_outputs.items():
        if path.exists():
            size_mb = path.stat().st_size / (1024 * 1024)
            logger.info(f"  ✓ {name} ({size_mb:.2f} MB)")
            print(f"  ✓ {name} ({size_mb:.2f} MB)")
else:
    logger.info("⚠️ No existing outputs - will run computation")
    print("⚠️ No existing outputs - will run computation")
    print(f"Output directory: {OUTPUT_DIR}")

### Runtime Measurement Function

In [ ]:
def measure_runtime(
    T: int,
    d: int,
    method: str,
    p_grid_max: int,
    n_jobs: int,
    seed: int = SEED,
) -> dict[str, float]:
    """Measure runtime for a single configuration.
    
    Parameters
    ----------
    T : int
        Time series length.
    d : int
        Dimensionality.
    method : str
        Method name ('cgc' or 'fcgc').
    p_grid_max : int
        Maximum p value for depth search.
    n_jobs : int
        Number of parallel jobs.
    seed : int
        Random seed.
    
    Returns
    -------
    dict[str, float]
        Dictionary with 'runtime_seconds' and 'error' (if any).
    """
    
    try:
        # Generate synthetic data
        scenario_result = scenario_order1_unconfounded(
            T=T,
            d=d,
            seed=seed + hash(f"{T}_{d}_{method}_{p_grid_max}_{n_jobs}") % 10000,
        )
        X = scenario_result.X
        
        # Create p_values list
        p_values = list(range(1, min(p_grid_max + 1, T // 10)))  # Limit p based on T
        if not p_values:
            p_values = [1]
        
        # Select analysis function
        if method == 'cgc':
            analyze_func = analyze_with_gcstar_cgc
        elif method == 'fcgc':
            analyze_func = analyze_with_gcstar_fcgc
        else:
            raise ValueError(f"Unknown method: {method}")
        
        # Measure runtime
        start_time = time.perf_counter()
        _ = analyze_func(X, p_values)
        end_time = time.perf_counter()
        
        runtime = end_time - start_time
        return {"runtime_seconds": runtime, "error": None}
        
    except Exception as e:
        return {"runtime_seconds": None, "error": str(e)}

print("✓ Measurement function defined")

### Collect Runtime Measurements

In [ ]:
if not outputs_exist:
    logger.info("Starting runtime measurements...")
    print("Starting runtime measurements...")
    print("This may take several minutes depending on data sizes and methods.\n")

    measurements = []
    total_combinations = (
        len(T_VALUES) * len(D_VALUES) * len(METHODS) * 
        len(P_GRID_MAX_VALUES) * len(N_JOBS_VALUES)
    )
    completed = 0
    
    start_time = time.time()

    for T in T_VALUES:
        for d in D_VALUES:
            for method in METHODS:
                for p_grid_max in P_GRID_MAX_VALUES:
                    for n_jobs in N_JOBS_VALUES:
                        completed += 1
                        status = f"[{completed}/{total_combinations}] "
                        config_str = f"T={T:5d}, d={d:2d}, {method:4s}, p_max={p_grid_max}, n_jobs={n_jobs}"
                        
                        print(f"{status}{config_str}", end=" ", flush=True)
                        logger.info(f"[{completed}/{total_combinations}] {config_str}")
                        
                        result = measure_runtime(
                            T=T,
                            d=d,
                            method=method,
                            p_grid_max=p_grid_max,
                            n_jobs=n_jobs,
                        )
                        
                        if result["error"] is None:
                            runtime = result["runtime_seconds"]
                            print(f"→ {runtime:.3f}s")
                            measurements.append({
                                "T": T,
                                "d": d,
                                "method": method,
                                "p_grid_max": p_grid_max,
                                "n_jobs": n_jobs,
                                "runtime_seconds": runtime,
                            })
                        else:
                            print(f"✗ Error: {result['error'][:40]}")
                            logger.warning(f"Error: {result['error'][:40]}")

    total_elapsed = time.time() - start_time
    logger.info(f"✓ Completed {len(measurements)} successful measurements in {total_elapsed:.1f}s")
    print(f"\n✓ Completed {len(measurements)} successful measurements")
else:
    logger.info("Skipping runtime measurements (loading from cache)")
    print("Skipping runtime measurements (loading from cache)")

In [ ]:
# Load cached measurements if they exist
if outputs_exist:
    runtime_path = OUTPUT_DIR / 'runtime_grid.csv'
    runtime_df = pd.read_csv(runtime_path)
    print(f"Loaded cached measurements: {len(runtime_df)} rows")
    print(f"\nSummary statistics (runtime_seconds):")
    print(runtime_df['runtime_seconds'].describe())
else:
    # Convert measurements to DataFrame (created in previous cell if computation ran)
    runtime_df = pd.DataFrame(measurements)

### Export Runtime Grid to CSV

In [ ]:
if not outputs_exist:
    logger.info("Exporting runtime measurements to CSV...")
    runtime_path = OUTPUT_DIR / 'runtime_grid.csv'
    runtime_df.to_csv(runtime_path, index=False)
    logger.info(f"Exported: {runtime_path}")
    print(f"Exported: {runtime_path}")

logger.info(f"DataFrame shape: {runtime_df.shape}")
logger.info(f"\nRuntime statistics (seconds):")
logger.info(f"\n{runtime_df['runtime_seconds'].describe().to_string()}")

print(f"\nDataFrame shape: {runtime_df.shape}")
print(f"\nSummary statistics (runtime_seconds):")
print(runtime_df['runtime_seconds'].describe())
print(f"\nFirst 10 rows:")
print(runtime_df.head(10).to_string())

### Create Scaling Visualization

In [ ]:
# Create visualization (always run for rendering)
logger.info("Generating runtime scaling visualizations...")
plot_start = time.time()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Runtime vs T (time series length) - by method
ax = axes[0, 0]
for method in METHODS:
    method_data = runtime_df[runtime_df['method'] == method]
    grouped = method_data.groupby('T')['runtime_seconds'].mean()
    ax.plot(grouped.index, grouped.values, marker='o', label=method, linewidth=2)
    logger.info(f"  Plotted {method}: runtime vs T")
ax.set_xlabel('Time Series Length (T)', fontsize=11)
ax.set_ylabel('Runtime (seconds)', fontsize=11)
ax.set_title('Runtime Scaling with Time Series Length', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Runtime vs d (dimensionality) - by method
ax = axes[0, 1]
for method in METHODS:
    method_data = runtime_df[runtime_df['method'] == method]
    grouped = method_data.groupby('d')['runtime_seconds'].mean()
    ax.plot(grouped.index, grouped.values, marker='s', label=method, linewidth=2)
    logger.info(f"  Plotted {method}: runtime vs d")
ax.set_xlabel('Dimensionality (d)', fontsize=11)
ax.set_ylabel('Runtime (seconds)', fontsize=11)
ax.set_title('Runtime Scaling with Dimensionality', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 3: Parallelization effect (n_jobs=1 vs n_jobs=4) - by method
ax = axes[1, 0]
for method in METHODS:
    for n_jobs in sorted(N_JOBS_VALUES):
        method_data = runtime_df[
            (runtime_df['method'] == method) & (runtime_df['n_jobs'] == n_jobs)
        ]
        grouped = method_data.groupby('T')['runtime_seconds'].mean()
        label = f"{method} (n_jobs={n_jobs})"
        style = '--' if n_jobs == 4 else '-'
        ax.plot(grouped.index, grouped.values, marker='o', label=label, 
                linestyle=style, linewidth=2, alpha=0.7)
logger.info("  Plotted parallelization effect")
ax.set_xlabel('Time Series Length (T)', fontsize=11)
ax.set_ylabel('Runtime (seconds)', fontsize=11)
ax.set_title('Parallelization Effect (n_jobs)', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Plot 4: Speedup vs n_jobs - by method and T
ax = axes[1, 1]
for method in METHODS:
    speedups = []
    T_vals = []
    for T in sorted(runtime_df['T'].unique()):
        method_T_data = runtime_df[
            (runtime_df['method'] == method) & (runtime_df['T'] == T)
        ]
        t1 = method_T_data[method_T_data['n_jobs'] == 1]['runtime_seconds'].mean()
        t4 = method_T_data[method_T_data['n_jobs'] == 4]['runtime_seconds'].mean()
        if t4 > 0 and t1 > 0:
            speedup = t1 / t4
            speedups.append(speedup)
            T_vals.append(T)
    if speedups:
        ax.plot(T_vals, speedups, marker='D', label=method, linewidth=2)

ax.axhline(y=1.0, color='gray', linestyle=':', linewidth=1, alpha=0.5)
ax.axhline(y=4.0, color='gray', linestyle=':', linewidth=1, alpha=0.5, label='Linear scaling')
ax.set_xlabel('Time Series Length (T)', fontsize=11)
ax.set_ylabel('Speedup (T=1 / T=4)', fontsize=11)
ax.set_title('Parallelization Speedup (n_jobs: 1 → 4)', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(bottom=0.5)
logger.info("  Plotted speedup analysis")

plt.tight_layout()
plot_path = OUTPUT_DIR / 'runtime_plot.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plot_elapsed = time.time() - plot_start
logger.info(f"✓ Visualizations completed in {plot_elapsed:.2f}s")
print(f"Exported: {plot_path}")
plt.close()

### Create Analysis Manifest

In [ ]:
if not outputs_exist:
    logger.info("Creating manifest file...")
    
    def _get_git_commit() -> str:
        """Get the current git commit hash."""
        try:
            import subprocess
            result = subprocess.run(
                ["git", "rev-parse", "--short", "HEAD"],
                capture_output=True,
                text=True,
                timeout=5,
                cwd=PROJECT_ROOT,
            )
            return result.stdout.strip() if result.returncode == 0 else "unknown"
        except Exception:
            return "unknown"

    manifest = {
        "created_at": datetime.now(timezone.utc).isoformat(),
        "git_commit": _get_git_commit(),
        "analysis": "runtime_scaling",
        "phase": "10.1-10.2",
        "output_paths": [
            str(runtime_path),
            str(plot_path),
        ],
        "measurement_grid": {
            "T_values": T_VALUES,
            "d_values": D_VALUES,
            "methods": METHODS,
            "p_grid_max_values": P_GRID_MAX_VALUES,
            "n_jobs_values": N_JOBS_VALUES,
            "total_combinations": len(T_VALUES) * len(D_VALUES) * len(METHODS) * len(P_GRID_MAX_VALUES) * len(N_JOBS_VALUES),
            "successful_measurements": len(runtime_df),
        },
        "random_seed": SEED,
        "scenario": "order1_unconfounded",
        "software_versions": {
            "python": f"{sys.version.split()[0]}",
            "numpy": f"{np.__version__}",
            "pandas": f"{pd.__version__}",
            "matplotlib": f"{plt.matplotlib.__version__}",
        },
        "runtime_statistics": {
            "min_seconds": float(runtime_df['runtime_seconds'].min()),
            "max_seconds": float(runtime_df['runtime_seconds'].max()),
            "mean_seconds": float(runtime_df['runtime_seconds'].mean()),
            "median_seconds": float(runtime_df['runtime_seconds'].median()),
        },
    }

    manifest_path = OUTPUT_DIR / 'manifest.json'
    with open(manifest_path, 'w') as f:
        json.dump(manifest, f, indent=2)

    logger.info(f"Exported manifest: {manifest_path}")
    logger.info(f"  Git commit: {manifest['git_commit']}")
    logger.info(f"  Successful measurements: {manifest['measurement_grid']['successful_measurements']}")
    logger.info(f"  Runtime range: {manifest['runtime_statistics']['min_seconds']:.3f}s - {manifest['runtime_statistics']['max_seconds']:.3f}s")
    
    print(f"Exported: {manifest_path}")
    print(f"\nManifest (summary):")
    print(f"  Git commit: {manifest['git_commit']}")
    print(f"  Successful measurements: {manifest['measurement_grid']['successful_measurements']}")
    print(f"  Runtime range: {manifest['runtime_statistics']['min_seconds']:.3f}s - {manifest['runtime_statistics']['max_seconds']:.3f}s")
else:
    logger.info("Loading manifest from cache")
    manifest_path = OUTPUT_DIR / 'manifest.json'
    print(f"Loading manifest from cache: {manifest_path}")
    with open(manifest_path) as f:
        manifest = json.load(f)

### Verification

In [ ]:
logger.info("Performing final verification...")
print("\nFinal Verification:")
expected_files = [
    runtime_path,
    plot_path,
    manifest_path,
]

all_exist = True
for fpath in expected_files:
    exists = fpath.exists()
    status = '✓' if exists else '✗'
    size = fpath.stat().st_size if exists else 'N/A'
    print(f"{status} {fpath.name:25s} ({size} bytes)")
    logger.info(f"{status} {fpath.name:25s} ({size} bytes)")
    all_exist = all_exist and exists

logger.info("CSV Data Summary:")
logger.info(f"  Total rows: {len(runtime_df)}")
logger.info(f"  Unique T values: {runtime_df['T'].nunique()}")
logger.info(f"  Unique d values: {runtime_df['d'].nunique()}")
logger.info(f"  Unique methods: {runtime_df['method'].nunique()}")
logger.info(f"  Unique p_grid_max: {runtime_df['p_grid_max'].nunique()}")
logger.info(f"  Unique n_jobs: {runtime_df['n_jobs'].nunique()}")

print(f"\nCSV Data Summary:")
print(f"  Total rows: {len(runtime_df)}")
print(f"  Unique T values: {runtime_df['T'].nunique()}")
print(f"  Unique d values: {runtime_df['d'].nunique()}")
print(f"  Unique methods: {runtime_df['method'].nunique()}")
print(f"  Unique p_grid_max: {runtime_df['p_grid_max'].nunique()}")
print(f"  Unique n_jobs: {runtime_df['n_jobs'].nunique()}")

if all_exist:
    logger.info("✓ Phase 10.1-10.2 complete: All outputs verified successfully")
    print("\n✓ Phase 10.1-10.2 complete: All outputs verified successfully")
else:
    logger.error("✗ Some outputs are missing")
    print("\n✗ Some outputs are missing")